In [1]:
import os

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
import requests

In [2]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_api_url = "https://api.anthropic.com/v1/"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_api_url)

In [ ]:
def fetch_website_contents(url: str) -> str | None:
    """Fetches the contents of a website given its URL."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return title + "\n\n" + text

In [4]:
def fetch_wikipedia_page(subject:str) -> str | None:
    url = f"https://en.wikipedia.org/wiki/{subject.lower().strip().replace(' ', '_').replace("-", "_")}"
    return fetch_website_contents(url)

In [5]:
city = "Paris"

In [6]:
page = fetch_wikipedia_page(city)

In [7]:
def summarize_text(text: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = """You are a helpful assistant that summarizes text in a very concise (less or equal to 2000 characters), structured and compelling way,
    ignoring text that might be navigation related. 
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Summarize the following text in a very concise (less or equal to 2000 characters), structured and compelling way:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    result = response.choices[0].message.content
    return result[:4096] if result else None

In [8]:
if page:
    summarized_page = summarize_text(page)

In [36]:
def summarize_wikipedia_page(subject: str) -> str | None:
    """Fetches the Wikipedia page for a given subject and summarizes its content."""
    print(f"TOOL CALL: summarize_wikipedia_page(subject={subject})")
    page = fetch_wikipedia_page(subject)
    if not page:
        return None
    summary = summarize_text(page)
    return summary

In [9]:
def find_out_the_main_spoken_language_in_city(city: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = """You are a helpful assistant that finds out the main language spoken in a city.
    Respond with the name of the language only, without any additional text."""
    user_prompt = f"What is the main language spoken in {city}?"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content

In [10]:
spoken_language = find_out_the_main_spoken_language_in_city(city)

In [11]:
def translate_text(text: str, target_language: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = f"""You are a helpful assistant that translates text into {target_language}.
    Respond with the translated text only, without any additional text.
    The response should contain 2000 characters at most.
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Translate the following text into {target_language}, in 2000 characters or less:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content[:4096] if response.choices[0].message.content else None

In [12]:
if summarized_page and spoken_language:
    translated_summary = translate_text(summarized_page, spoken_language)

In [38]:
def translate_text_into_city_language(text: str, city: str) -> str | None:
    """Translates the given text into the main language spoken in the specified city."""
    print(f"TOOL CALL: translate_text_into_city_language(text={text}, city={city})")
    main_language = find_out_the_main_spoken_language_in_city(city)
    if not main_language:
        return None
    translated_text = translate_text(text, main_language)
    return translated_text

In [37]:
def talker(text: str, model: str = "tts-1") -> bytes | None:
    """Generates speech from the given text using OpenAI's TTS model."""
    print(f"TOOL CALL: talker(text={text}, model={model})")
    response = openai.audio.speech.create(model=model, voice="coral", input=text[:4096], speed=1)
    return response.content

In [ ]:
# from IPython.display import Audio, display

# if translated_summary:
#     audio_bytes = talker(translated_summary)
#     display(Audio(audio_bytes, autoplay=True))

In [15]:
def fetch_wikipedia_image_url(subject: str) -> str | None:
    """Fetches the first image URL from the Wikipedia page of the given subject."""
    normalized_subject = subject.lower().strip().replace(" ", "_").replace("-", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{normalized_subject}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    image = response.json().get("originalimage")
    return image["source"] if image else None

In [29]:
def get_function_name_and_description(func) -> tuple[str, str | None]:
    name = func.__name__
    description = func.__doc__ or None
    return (name, description)

In [ ]:
def turn_function_into_tool(func, required_parameters):
    name, description = get_function_name_and_description(func)
    function_dictionary = (
        {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": required_parameters,
                "required": list(required_parameters.keys()),
                "additionalProperties": False,
            },
        },
    )
    return {"type": "function", "function": function_dictionary}


# Parameters

In [ ]:
fetch_wikipedia_image_url_params = {
    "subject": {
        "type": "string",
        "description": "The subject of the Wikipedia page to fetch the first image from.",
    }
}

summarize_wikipedia_page_params = {
    "subject": {
        "type": "string",
        "description": "The subject of the Wikipedia page to summarize.",
    }
}

translate_text_into_city_language_params = {
    "text": {
        "type": "string",
        "description": "The text to translate into the main language spoken in the specified city.",
    },
    "city": {
        "type": "string",
        "description": "The city whose main language will be used for translation.",
    },
}

talker_params = {
    "text": {
        "type": "string",
        "description": "The text to convert into speech.",
    }
}


# Tools

In [40]:
fetch_wikipedia_image_url_tool = turn_function_into_tool(
    fetch_wikipedia_image_url, fetch_wikipedia_image_url_params
)
summarize_wikipedia_page_tool = turn_function_into_tool(
    summarize_wikipedia_page, summarize_wikipedia_page_params
)
translate_text_into_city_language_tool = turn_function_into_tool(
    translate_text_into_city_language, translate_text_into_city_language_params
)
talker_tool = turn_function_into_tool(talker, talker_params)
